# CartIQ — Phase 4: Neural Collaborative Filtering (NCF)
**CS Machine Learning Course Project — Group 3**

Implements NCF with user/product embeddings + MLP to learn latent interaction patterns. Trained with PyTorch.

In [ ]:
!pip install -q torch scikit-learn matplotlib seaborn

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os, time, warnings
warnings.filterwarnings('ignore')

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import precision_score, recall_score, f1_score, roc_auc_score, classification_report

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
sns.set_style('whitegrid')

In [ ]:
# Load data
train_df = pd.read_parquet('data/train.parquet')
val_df = pd.read_parquet('data/val.parquet')
test_df = pd.read_parquet('data/test.parquet')

# Create contiguous ID mappings for embeddings
all_users = pd.concat([train_df['user_id'], val_df['user_id'], test_df['user_id']]).unique()
all_products = pd.concat([train_df['product_id'], val_df['product_id'], test_df['product_id']]).unique()

user2idx = {uid: i for i, uid in enumerate(all_users)}
prod2idx = {pid: i for i, pid in enumerate(all_products)}

n_users = len(user2idx)
n_products = len(prod2idx)

print(f'Users: {n_users:,} | Products: {n_products:,}')
print(f'Train: {len(train_df):,} | Val: {len(val_df):,} | Test: {len(test_df):,}')

## 4.1 NCF Dataset with Negative Sampling

In [ ]:
class NCFDataset(Dataset):
    """Dataset for NCF with negative sampling (1:4 ratio)."""
    
    def __init__(self, df, user2idx, prod2idx, neg_ratio=4):
        self.user2idx = user2idx
        self.prod2idx = prod2idx
        
        # Positive samples
        positives = df[df['label'] == 1][['user_id', 'product_id']].values
        negatives = df[df['label'] == 0][['user_id', 'product_id']].values
        
        # Subsample negatives to neg_ratio * positives
        n_neg = min(len(negatives), len(positives) * neg_ratio)
        neg_idx = np.random.RandomState(42).choice(len(negatives), size=n_neg, replace=False)
        negatives = negatives[neg_idx]
        
        # Combine
        self.users = np.concatenate([
            np.array([user2idx[u] for u in positives[:, 0]]),
            np.array([user2idx[u] for u in negatives[:, 0]]),
        ])
        self.products = np.concatenate([
            np.array([prod2idx[p] for p in positives[:, 1]]),
            np.array([prod2idx[p] for p in negatives[:, 1]]),
        ])
        self.labels = np.concatenate([
            np.ones(len(positives)),
            np.zeros(len(negatives)),
        ])
        
        # Shuffle
        perm = np.random.RandomState(42).permutation(len(self.labels))
        self.users = self.users[perm]
        self.products = self.products[perm]
        self.labels = self.labels[perm]
        
        print(f'  Samples: {len(self.labels):,} (pos: {int(self.labels.sum()):,}, neg: {int((1-self.labels).sum()):,})')
    
    def __len__(self):
        return len(self.labels)
    
    def __getitem__(self, idx):
        return (
            torch.tensor(self.users[idx], dtype=torch.long),
            torch.tensor(self.products[idx], dtype=torch.long),
            torch.tensor(self.labels[idx], dtype=torch.float32),
        )

print('Building datasets...')
print('Train:')
train_ds = NCFDataset(train_df, user2idx, prod2idx, neg_ratio=4)
print('Val:')
val_ds = NCFDataset(val_df, user2idx, prod2idx, neg_ratio=4)
print('Test:')
test_ds = NCFDataset(test_df, user2idx, prod2idx, neg_ratio=4)

train_loader = DataLoader(train_ds, batch_size=1024, shuffle=True, num_workers=0)
val_loader = DataLoader(val_ds, batch_size=2048, shuffle=False, num_workers=0)
test_loader = DataLoader(test_ds, batch_size=2048, shuffle=False, num_workers=0)

## 4.2 NCF Model Architecture
`Embedding(user, 64) + Embedding(product, 64) → concat → MLP [256, 128, 64] → Sigmoid`

In [ ]:
class NCF(nn.Module):
    """Neural Collaborative Filtering."""
    
    def __init__(self, n_users, n_items, embed_dim=64, dropout=0.3):
        super().__init__()
        self.user_embed = nn.Embedding(n_users, embed_dim)
        self.item_embed = nn.Embedding(n_items, embed_dim)
        
        self.mlp = nn.Sequential(
            nn.Linear(embed_dim * 2, 256),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(64, 1),
        )
        
        # Initialize
        nn.init.xavier_uniform_(self.user_embed.weight)
        nn.init.xavier_uniform_(self.item_embed.weight)
    
    def forward(self, user_ids, item_ids):
        u = self.user_embed(user_ids)
        i = self.item_embed(item_ids)
        x = torch.cat([u, i], dim=-1)
        return self.mlp(x).squeeze(-1)

model = NCF(n_users, n_products, embed_dim=64, dropout=0.3).to(device)
total_params = sum(p.numel() for p in model.parameters())
print(f'NCF Model — {total_params:,} parameters')
print(model)

## 4.3 Training Loop

In [ ]:
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-5)
criterion = nn.BCEWithLogitsLoss()

EPOCHS = 20
PATIENCE = 3
best_val_f1 = 0
patience_counter = 0
history = {'train_loss': [], 'val_loss': [], 'val_f1': [], 'val_auc': []}

print(f'Training NCF for up to {EPOCHS} epochs (patience={PATIENCE})...\n')
t0 = time.time()

for epoch in range(EPOCHS):
    # Train
    model.train()
    train_losses = []
    for users, items, labels in train_loader:
        users, items, labels = users.to(device), items.to(device), labels.to(device)
        optimizer.zero_grad()
        logits = model(users, items)
        loss = criterion(logits, labels)
        loss.backward()
        optimizer.step()
        train_losses.append(loss.item())
    
    avg_train_loss = np.mean(train_losses)
    
    # Validate
    model.eval()
    val_losses, val_preds, val_labels = [], [], []
    with torch.no_grad():
        for users, items, labels in val_loader:
            users, items, labels = users.to(device), items.to(device), labels.to(device)
            logits = model(users, items)
            val_losses.append(criterion(logits, labels).item())
            val_preds.append(torch.sigmoid(logits).cpu().numpy())
            val_labels.append(labels.cpu().numpy())
    
    val_preds = np.concatenate(val_preds)
    val_labels = np.concatenate(val_labels)
    avg_val_loss = np.mean(val_losses)
    val_f1 = f1_score(val_labels, (val_preds >= 0.5).astype(int))
    val_auc = roc_auc_score(val_labels, val_preds)
    
    history['train_loss'].append(avg_train_loss)
    history['val_loss'].append(avg_val_loss)
    history['val_f1'].append(val_f1)
    history['val_auc'].append(val_auc)
    
    print(f'Epoch {epoch+1:2d}/{EPOCHS} — train_loss: {avg_train_loss:.4f} | val_loss: {avg_val_loss:.4f} | val_F1: {val_f1:.4f} | val_AUC: {val_auc:.4f}')
    
    # Early stopping
    if val_f1 > best_val_f1:
        best_val_f1 = val_f1
        patience_counter = 0
        torch.save(model.state_dict(), 'models/ncf_best.pt')
    else:
        patience_counter += 1
        if patience_counter >= PATIENCE:
            print(f'\nEarly stopping at epoch {epoch+1}')
            break

train_time = time.time() - t0
print(f'\nTraining completed in {train_time:.1f}s')
print(f'Best val F1: {best_val_f1:.4f}')

In [ ]:
# Training curves
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

axes[0].plot(history['train_loss'], label='Train', color='#3b82f6')
axes[0].plot(history['val_loss'], label='Val', color='#ef4444')
axes[0].set_title('Loss', fontweight='bold')
axes[0].legend()

axes[1].plot(history['val_f1'], color='#10b981')
axes[1].set_title('Validation F1', fontweight='bold')

axes[2].plot(history['val_auc'], color='#f59e0b')
axes[2].set_title('Validation ROC-AUC', fontweight='bold')

for ax in axes:
    ax.set_xlabel('Epoch')

plt.tight_layout()
plt.savefig('ncf_training_curves.png', dpi=150, bbox_inches='tight')
plt.show()

## 4.4 Evaluate NCF on Test Set

In [ ]:
# Load best model
model.load_state_dict(torch.load('models/ncf_best.pt', map_location=device))
model.eval()

test_preds, test_labels_all = [], []
with torch.no_grad():
    for users, items, labels in test_loader:
        users, items = users.to(device), items.to(device)
        logits = model(users, items)
        test_preds.append(torch.sigmoid(logits).cpu().numpy())
        test_labels_all.append(labels.numpy())

ncf_proba = np.concatenate(test_preds)
ncf_labels = np.concatenate(test_labels_all)
ncf_pred = (ncf_proba >= 0.5).astype(int)

ncf_metrics = {
    'model': 'NCF',
    'precision': precision_score(ncf_labels, ncf_pred),
    'recall': recall_score(ncf_labels, ncf_pred),
    'f1': f1_score(ncf_labels, ncf_pred),
    'roc_auc': roc_auc_score(ncf_labels, ncf_proba),
}

print('=== Neural Collaborative Filtering ===')
print(f'Precision: {ncf_metrics["precision"]:.4f}')
print(f'Recall:    {ncf_metrics["recall"]:.4f}')
print(f'F1-Score:  {ncf_metrics["f1"]:.4f}')
print(f'ROC-AUC:   {ncf_metrics["roc_auc"]:.4f}')
print()
print(classification_report(ncf_labels, ncf_pred, target_names=['Not Reordered', 'Reordered']))

## 4.5 Product Embedding Visualization (t-SNE)

In [ ]:
from sklearn.manifold import TSNE

# Get product embeddings
products_info = pd.read_parquet('data/products.parquet')

# Get embeddings for products that are in our index
prod_embeds = model.item_embed.weight.detach().cpu().numpy()

# Map product indices back to product IDs and get department info
idx2prod = {v: k for k, v in prod2idx.items()}
embed_df = pd.DataFrame({
    'product_id': [idx2prod[i] for i in range(min(5000, len(idx2prod)))],
})
embed_df = embed_df.merge(products_info[['product_id', 'department']], on='product_id', how='left')

# Take top 5000 for visualization
n_viz = len(embed_df)
embeddings_subset = prod_embeds[:n_viz]

print(f'Running t-SNE on {n_viz} product embeddings...')
tsne = TSNE(n_components=2, random_state=42, perplexity=30)
coords = tsne.fit_transform(embeddings_subset)

embed_df['x'] = coords[:, 0]
embed_df['y'] = coords[:, 1]

# Plot top departments
top_depts = embed_df['department'].value_counts().head(8).index.tolist()
plot_df = embed_df[embed_df['department'].isin(top_depts)]

fig, ax = plt.subplots(figsize=(12, 8))
for dept in top_depts:
    mask = plot_df['department'] == dept
    ax.scatter(plot_df.loc[mask, 'x'], plot_df.loc[mask, 'y'],
               s=8, alpha=0.5, label=dept)

ax.set_title('Product Embeddings (t-SNE) — Colored by Department', fontweight='bold', fontsize=14)
ax.legend(markerscale=3, fontsize=10, loc='best')
ax.set_xticks([])
ax.set_yticks([])
plt.tight_layout()
plt.savefig('ncf_embeddings_tsne.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Save metrics
pd.DataFrame([ncf_metrics]).to_csv('models/metrics_ncf.csv', index=False)
print('NCF metrics saved.')